# VLM3D Multi-Abnormality 3D Chest CT Classification

This notebook provides a complete end-to-end pipeline for the VLM3D Challenge: Task 1 (Multi Abnormality Classification). It processes 3D CT-RATE `.nii.gz` volumes, dynamically handles profound class imbalances across 18 pathologies, applies MONAI 3D augmentations, and trains a 3D ResNet. The training loop integrates `BCEWithLogitsLoss`, `CosineAnnealingLR`, and logs `Macro AUROC`/`Macro F1` metrics via TensorBoard.

In [ ]:
!pip install -q "monai[all]==1.3.2" simpleitk nibabel torchmetrics tensorboard

In [ ]:
import os
import csv
import torch
import torch.nn as nn
import numpy as np
from monai.networks.nets import resnet10
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, CropForegroundd, ResizeWithPadOrCropd,
    RandFlipd, RandAffined, RandGaussianNoised, RandScaleIntensityd, RandShiftIntensityd,
    EnsureTyped
)
from monai.data import DataLoader, Dataset
from monai.utils import set_determinism
from torchmetrics.classification import MultilabelAUROC, MultilabelF1Score
from torch.utils.tensorboard import SummaryWriter

# Set deterministic training for reproducibility
set_determinism(seed=42)

## 1. Data Loader & Preprocessing
Parse the `train_labels.csv` to MONAI metadata dictionaries and apply transforms.

In [ ]:
def parse_dataset_csv(csv_path, images_dir):
    data_dicts = []
    if not os.path.exists(csv_path):
        print(f"Warning: CSV {csv_path} not found.")
        return data_dicts
        
    with open(csv_path, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            vol_name = row[0]
            labels = np.array([float(x) for x in row[1:]], dtype=np.float32)
            data_dicts.append({
                "image": os.path.join(images_dir, vol_name),
                "label": labels
            })
    return data_dicts

def get_transforms(mode="train"):
    base_transforms = [
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"]),
        Spacingd(keys=["image"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear")),
        Orientationd(keys=["image"], axcodes="RAS"),
        ScaleIntensityRangePercentilesd(keys=["image"], lower=5.0, upper=95.0, b_min=0.0, b_max=1.0, clip=True),
        CropForegroundd(keys=["image"], source_key="image"),
        ResizeWithPadOrCropd(keys=["image"], spatial_size=(128, 128, 64)),
    ]
    
    if mode == "train":
        train_transforms = [
            RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
            RandFlipd(keys=["image"], prob=0.5, spatial_axis=1),
            RandFlipd(keys=["image"], prob=0.5, spatial_axis=2),
            RandAffined(keys=["image"], prob=0.3, rotate_range=(0.1, 0.1, 0.1), scale_range=(0.1, 0.1, 0.1)),
            RandGaussianNoised(keys=["image"], prob=0.1),
            RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.1),
            RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.1)
        ]
        transforms = base_transforms + train_transforms
    else:
        transforms = base_transforms

    transforms.append(EnsureTyped(keys=["image", "label"]))
    return Compose(transforms)

def get_dataloaders(train_csv, valid_csv, images_dir, batch_size=4):
    train_data = parse_dataset_csv(train_csv, images_dir) if train_csv else []
    val_data = parse_dataset_csv(valid_csv, images_dir) if valid_csv else []
    
    train_loader = None
    if train_data:
        train_dataset = Dataset(data=train_data, transform=get_transforms("train"))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
        print(f"Training samples loaded: {len(train_data)}")
        
    val_loader = None
    if val_data:
        val_dataset = Dataset(data=val_data, transform=get_transforms("val"))
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
        print(f"Validation samples loaded: {len(val_data)}")
        
    return train_loader, val_loader, train_data

## 2. Model Architecture & Class Weight Optimizer
Calculating robust positive weights to combat CT-RATE structural imbalances.

In [ ]:
class ChestCTClassificationModel(nn.Module):
    def __init__(self, num_classes=18, spatial_dims=3, n_input_channels=1):
        super(ChestCTClassificationModel, self).__init__()
        self.backbone = resnet10(
            spatial_dims=spatial_dims,
            n_input_channels=n_input_channels,
            num_classes=num_classes
        )
    
    def forward(self, x):
        return self.backbone(x)

def calculate_pos_weight(data_dicts, num_classes=18):
    total_samples = len(data_dicts)
    if total_samples == 0:
        return torch.ones(num_classes)
        
    counts = np.zeros(num_classes)
    for d in data_dicts:
        counts += d["label"]
        
    pos_weights = []
    for c in counts:
        if c > 0:
            pw = (total_samples - c) / c
            pw = min(pw, 50.0)  # Ceiling limit to avoid exploding gradients
        else:
            pw = 1.0
        pos_weights.append(pw)
    return torch.tensor(pos_weights, dtype=torch.float32)

## 3. Training & Evaluation Pipeline

In [ ]:
def train_and_evaluate(train_csv, valid_csv, images_dir, num_epochs=10, batch_size=4, lr=1e-4, log_dir="runs/ct_classifier"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    writer = SummaryWriter(log_dir=log_dir)
    
    train_loader, val_loader, train_data = get_dataloaders(train_csv, valid_csv, images_dir, batch_size=batch_size)
    if not train_loader or not val_loader:
        print("Dataloaders could not be initialized completely. Ensure valid CSV formats and paths.")
        return

    # Pos Weights handling class imbalance dynamically
    pos_weight = calculate_pos_weight(train_data).to(device)
    print(f"Dynamic BCE pos_weights mapped per class:\n{pos_weight}\n")
    
    model = ChestCTClassificationModel(num_classes=18).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    auroc = MultilabelAUROC(num_labels=18, average="macro").to(device)
    f1_score = MultilabelF1Score(num_labels=18, average="macro").to(device)
    
    best_val_score = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        for step, batch in enumerate(train_loader):
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels.float())
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            global_step = epoch * len(train_loader) + step
            writer.add_scalar("Train/Loss_step", loss.item(), global_step)
            
        train_loss /= len(train_loader)
        writer.add_scalar("Train/Loss_epoch", train_loss, epoch)
        
        # Validation Phase
        model.eval()
        val_loss = 0.0
        auroc.reset()
        f1_score.reset()
        
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch["image"].to(device)
                labels = batch["label"].to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels.float())
                val_loss += loss.item()
                
                auroc.update(outputs, labels.long())
                f1_score.update(outputs, labels.long())
                
        val_loss /= len(val_loader)
        val_auroc = auroc.compute().item()
        val_f1 = f1_score.compute().item()
        
        scheduler.step()
        
        writer.add_scalar("Val/Loss_epoch", val_loss, epoch)
        writer.add_scalar("Val/AUROC_macro", val_auroc, epoch)
        writer.add_scalar("Val/F1_macro", val_f1, epoch)
        writer.add_scalar("Train/LR", scheduler.get_last_lr()[0], epoch)
        
        print(f"Epoch [{(epoch+1):02d}/{num_epochs}] Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val AUROC: {val_auroc:.4f} | Val F1: {val_f1:.4f}")
        
        score = val_auroc + val_f1
        if score > best_val_score:
            best_val_score = score
            torch.save(model.state_dict(), "best_classification_model.pth")
            print("=> Saved new best model checkpoint!")

    writer.close()
    print("Training Complete. Metrics logged to TensorBoard.")

## 4. Run Execution Block
Define your path structures relative to this notebook.

In [ ]:
# Adjust paths as necessary for your local or HPC cluster filesystem.
TRAIN_CSV = "dataset/example_download_script-main/train_labels.csv"
VALID_CSV = "dataset/example_download_script-main/valid_labels.csv"
IMAGES_DIR = "dataset/images"  # Update this to your raw NIfTI files path

# To execute the robust training loop, uncomment below:
# train_and_evaluate(TRAIN_CSV, VALID_CSV, IMAGES_DIR, num_epochs=20, batch_size=4)

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir runs/ct_classifier